<h2> Tool Defination</h2>

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
#function name is tool name and tool defination is defined in doc strings at beginging of function

@tool
def square_root(x:float)-> float:
    """Calculate the square root of number"""
    return(x**0.5)

In [ ]:
#we can over ride the actual defination of tool like following
@tool("square")
def tool1(x:float)->float:
    """Calculates square of number"""
    return x*x

In [4]:
@tool("even_R_odd",description="checks given number is even or odd and return true or false")
def tool2(x:int):
    if x//2 ==0:
        res=str(x)+'is even'
        return(res)
    else:
        res=str(x)+'is odd'
        return(res)
    

In [ ]:
print(tool1.args)
print(tool2.invoke({'x':4}))
print(square_root.invoke({'x':9}))

<h2>Adding Tools to Agents</h2>

In [ ]:
# to know available groq models
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

openai/gpt-oss-20b
allam-2-7b
whisper-large-v3-turbo
whisper-large-v3
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-86m


In [10]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    
)

agent=create_agent(
    model=model,
    tools=[tool1,tool2,square_root],
    system_prompt="You are an arithmetic agent. Use your tools to calculate the square root and square and even or odd of any number."
)

response=agent.invoke(
    {"messages":[HumanMessage(content="What is the square root and square of 81 and is it even or odd")]}
)

print(response['messages'][-1].content)

- **Square root of 81:** \( \sqrt{81} = 9 \)  
- **Square of 81:** \( 81^2 = 6561 \)  
- **Is 81 even or odd?** 81 is **odd**.


In [11]:
from pprint import pprint

pprint(response['messages'])
print("******************************")
print(response["messages"][1].tool_calls)

[HumanMessage(content='What is the square root and square of 81 and is it even or odd', additional_kwargs={}, response_metadata={}, id='612985ee-4df5-4ed5-92aa-5f9ae72d1731'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the square root and square of 81 and is it even or odd". They want square root, square, and even/odd. We need to use tools to calculate. We need to call functions: square_root, square, even_R_odd. We\'ll call each with x=81. Then we produce answer.\n\nWe need to use the function calls. We\'ll produce calls in separate messages? In this format, we can call multiple functions in one message? Usually, you call one function at a time. But we can chain calls: first square_root, then square, then even_R_odd. Let\'s do that. We\'ll call square_root(81). Then square(81). Then even_R_odd(81). We\'ll produce final answer.', 'tool_calls': [{'id': 'fc_43e0ec43-5489-4525-ba05-7355b5149daf', 'function': {'arguments': '{"x":81}', 'name': 'sq